In [1]:
!python --version
!nvcc --version
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Python 3.12.13
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp3hc3h1km".


In [2]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5


## Rozwiązanie

### Algorytm z wykorzystaniem pamięci globalnej

In [3]:
!nvcc matrixmul.cu -o matrixmul
!./matrixmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Average Execution Time [ 512x 512] over 20 runs : 1.020 ms
Average Execution Time [1024x1024] over 20 runs : 7.991 ms
Average Execution Time [2048x2048] over 20 runs : 36.765 ms


Dla macierzy B i C występuje pełny dostęp łączony, natomiast dla macierzy A optymalny dostęp realizowany jest poprzez mechanizm zrównoleglonego rozsyłania danych (broadcast) w obrębie jednego warpu.

Wzór na liczbę bloków realizuje dzielenie z zaokrągleniem w górę co gwarantuje wygenerowanie wystarczającej liczby bloków do pokrycia całej macierzy, wliczając w to jej skrajne elementy.

Typ `dim3` zmiennej reprezentujący wektor 3D z liczbami całkowitymi (x, y, z), który służy do wygodnego deklarowania wymiarów siatki i bloku wątków.

### Algorytm z wykorzystaniem pamięci współdzielonej

In [4]:
!nvcc matrixmul_shared.cu -o matrixmul_shared
!./matrixmul_shared

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Avg Shared Exec Time [ 512x 512] over 20 runs : 0.653 ms (Valid)
Avg Shared Exec Time [1024x1024] over 20 runs : 4.608 ms (Valid)
Avg Shared Exec Time [2048x2048] over 20 runs : 24.986 ms (Valid)


`TILE_DIM` odpowiada wymiarom bloku, ponieważ chcemy, aby każdy wątek wewnątrz bloku był odpowiedzialny za skopiowanie dokładnie jednego elementu podmacierzy (kafelka) z wolnej pamięci globalnej do szybkiej pamięci współdzielonej (Shared Memory).

Zgodnie ze specyfikacją architektury CUDA, pojedynczy blok może zawierać maksymalnie 1024 wątki. W przypadku siatki dwuwymiarowej o równych bokach oznacza to wymiary 32x32 (ponieważ 32 * 32 = 1024). W związku z tym największą wartością dla `blockDim.x` oraz `blockDim.y` w tym algorytmie jest 32.

Konflikty nie występują tutaj podczas dostępu do pamięci współdzielonej. Podczas fazy ładowania wątki z tego samego warpu pobierają kolejne wartości dla macierzy, trafiając do różnych banków pamięci. Podczas fazy mnożenia wartości `s_A` są jednakowe dla całego warpu w danej iteracji, co karta graficzna optymalizuje poprzez mechanizm rozsyłania (broadcast), zapobiegając konfliktom. Z kolei odpytywanie `s_B` odbywa się po ciągłych indeksach, co trafia do osobnych banków.

### Algorytm z wykorzystaniem biblioteki cuBLAS (w dwóch wariantach)

In [5]:
!nvcc matrixmul_cublas.cu -lcublas -o matrixmul_cublas
!./matrixmul_cublas

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
cuBLAS [ 512x 512] | Cold Start :   7.547 ms | Avg Warm (20 runs) : 0.095 ms
cuBLAS [1024x1024] | Cold Start :   0.622 ms | Avg Warm (20 runs) : 0.521 ms
cuBLAS [2048x2048] | Cold Start :   3.731 ms | Avg Warm (20 runs) : 3.648 ms


Modyfikacja w drugim wariancie polega na dodaniu pustego przebiegu warm-up. Funkcja mnożąca cublasSgemm jest wywoływana po raz pierwszy przed uruchomieniem zegara mierzącego czas.

Pierwsze wywołanie funkcji API z biblioteki takiej jak cuBLAS wiąże się z narzutem czasowym na inicjalizację kontekstu CUDA, wczytanie bibliotek do pamięci karty oraz alokację zasobów sprzętowych. Odizolowanie tego procesu w fazie "warm-up" pozwala zmierzyć rzeczywisty i stabilniejszy czas samych obliczeń.

## Podsumowanie

| Rozmiar Macierzy | Pamięć Globalna (Średnia z 20) | Pamięć Współdzielona (Średnia z 20) | cuBLAS (Cold) | cuBLAS (Średnia Warm z 20) |
| --- | --- | --- | --- | --- |
| **512 x 512** | 1.020 ms | 0.653 ms | 7.547 ms | 0.095 ms |
| **1024 x 1024** | 7.991 ms | 4.608 ms | 0.622 ms | 0.521 ms |
| **2048 x 2048** | 36.765 ms | 24.986 ms | 3.731 ms | 3.648 ms |